# 🛡️ DeepGuard – EfficientNet-B0 Model Training Notebook
### AI-Based Deepfake Image & Video Detection System

This notebook trains an **EfficientNet-B0** model on **Google Colab (with free T4 GPU)** for binary deepfake classification (`0: Real`, `1: Fake`).

At the conclusion of training, it exports **`deepguard_effnet.pth`**, ready for plug-and-play inference in DeepGuard's FastAPI backend (`backend/weights/deepguard_effnet.pth`).

## 1. Verify GPU Hardware Acceleration

In [1]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Sat Nov 14 10:14:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Required Dependencies

In [2]:
!pip install -q timm scikit-learn matplotlib torchvision

In [3]:
import os, glob, random, shutil, time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from google.colab import files

# 1. Dataset check (if runtime restarted, upload kaggle.json once)
if not os.path.exists("dataset/train/real"):
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        uploaded = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip install -q kaggle timm
    !kaggle datasets download -d xhlulu/140k-real-and-fake-faces
    !unzip -q 140k-real-and-fake-faces.zip -d raw_kaggle
    os.makedirs("dataset/train", exist_ok=True)
    os.makedirs("dataset/val", exist_ok=True)
    shutil.move(glob.glob("raw_kaggle/**/train/real", recursive=True)[0], "dataset/train/real")
    shutil.move(glob.glob("raw_kaggle/**/train/fake", recursive=True)[0], "dataset/train/fake")
    shutil.move(glob.glob("raw_kaggle/**/valid/real", recursive=True)[0], "dataset/val/real")
    shutil.move(glob.glob("raw_kaggle/**/valid/fake", recursive=True)[0], "dataset/val/fake")
    !rm -rf raw_kaggle 140k-real-and-fake-faces.zip

# 2. Fast balanced subset (10,000 train + 2,000 val) -> avoids Colab DataLoader freeze!
os.makedirs("fast_ds/train/real", exist_ok=True)
os.makedirs("fast_ds/train/fake", exist_ok=True)
os.makedirs("fast_ds/val/real", exist_ok=True)
os.makedirs("fast_ds/val/fake", exist_ok=True)

for cls in ["real", "fake"]:
    for split, count in [("train", 5000), ("val", 1000)]:
        src_files = glob.glob(f"dataset/{split}/{cls}/*")
        for f in random.sample(src_files, min(count, len(src_files))):
            dst = f"fast_ds/{split}/{cls}/{os.path.basename(f)}"
            if not os.path.exists(dst):
                os.symlink(os.path.abspath(f), dst)

# 3. Fast GPU Training (num_workers=0 prevents hang, batch_size=128 runs in ~45 secs!)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder("fast_ds/train", transform=tf)
val_ds = datasets.ImageFolder("fast_ds/val", transform=tf)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0)

model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

print(f"Training on {device} | Classes: {train_ds.class_to_idx}")
for epoch in range(1, 4):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    print(f"Epoch {epoch}/3 -> Val Accuracy: {100*correct/total:.2f}%")

torch.save(model.state_dict(), "deepguard_effnet.pth")
print("✅ Saved deepguard_effnet.pth! Downloading now...")
files.download("deepguard_effnet.pth")

Training on cuda | Classes: {'fake': 0, 'real': 1}
Epoch 1/3 -> Val Accuracy: 95.10%
Epoch 2/3 -> Val Accuracy: 97.40%
Epoch 3/3 -> Val Accuracy: 97.35%
✅ Saved deepguard_effnet.pth! Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
from google.colab import files
import os, shutil, glob

uploaded = files.upload()
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!rm -rf dataset
!pip install -q kaggle
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip -d raw_kaggle

os.makedirs("dataset/train", exist_ok=True)
os.makedirs("dataset/val", exist_ok=True)
shutil.move(glob.glob("raw_kaggle/**/train/real", recursive=True)[0], "dataset/train/real")
shutil.move(glob.glob("raw_kaggle/**/train/fake", recursive=True)[0], "dataset/train/fake")
shutil.move(glob.glob("raw_kaggle/**/valid/real", recursive=True)[0], "dataset/val/real")
shutil.move(glob.glob("raw_kaggle/**/valid/fake", recursive=True)[0], "dataset/val/fake")
!rm -rf raw_kaggle 140k-real-and-fake-faces.zip
print("✅ Ready! Train real:", len(os.listdir("dataset/train/real")))

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [03:21<00:00, 19.9MB/s]

✅ Ready! Train real: 50000


## 3. Dataset Setup (Kaggle '140k Real & Fake Faces' or Custom)

Ensure dataset is extracted with this folder hierarchy:
```
dataset/
├── train/
│   ├── real/
│   └── fake/
└── val/
    ├── real/
    └── fake/
```

To download directly from Kaggle in Colab:
```bash
!pip install kaggle
# Upload your kaggle.json to ~/.kaggle/kaggle.json
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip -d dataset/
```

*Note: The cell below will auto-generate a synthetic test dataset if no dataset exists yet so you can run the pipeline immediately.*

In [5]:
import os
from pathlib import Path
from PIL import Image

dataset_dir = Path("dataset")
train_dir = dataset_dir / "train"
val_dir = dataset_dir / "val"

if not train_dir.exists() or not val_dir.exists():
    print("Dataset directory not found. Creating synthetic placeholder dataset...")
    for split in ["train", "val"]:
        for category in ["real", "fake"]:
            folder = dataset_dir / split / category
            folder.mkdir(parents=True, exist_ok=True)
            count = 40 if split == "train" else 10
            for i in range(count):
                color = (50, 160, 80) if category == "real" else (190, 50, 50)
                img = Image.new("RGB", (224, 224), color=color)
                img.save(folder / f"sample_{i:04d}.jpg")
    print("Synthetic dataset ready at ./dataset/")
else:
    print(f"Found existing dataset at {dataset_dir.resolve()}")

Found existing dataset at /content/dataset


## 4. Data Transforms & Strict Binary DataLoaders

In [6]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from typing import Tuple, Dict

class StrictBinaryFolder(datasets.ImageFolder):
    """Strictly enforces Class 0 = Real, Class 1 = Fake"""
    def find_classes(self, directory: str) -> Tuple[list, Dict[str, int]]:
        classes, _ = super().find_classes(directory)
        class_to_idx = {}
        for c in classes:
            low = c.lower()
            if "real" in low:
                class_to_idx[c] = 0
            elif "fake" in low or "synth" in low:
                class_to_idx[c] = 1
            else:
                raise ValueError(f"Unknown folder '{c}'. Expected 'real' or 'fake'.")
        ordered_classes = sorted(class_to_idx.keys(), key=lambda k: class_to_idx[k])
        return ordered_classes, class_to_idx

# Alias for compatibility
StrictBinaryImageFolder = StrictBinaryFolder

train_transform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = StrictBinaryImageFolder("dataset/train", transform=train_transform)
val_dataset = StrictBinaryImageFolder("dataset/val", transform=val_transform)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")
print(f"Class mapping: {train_dataset.class_to_idx}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

Train samples: 100000, Validation samples: 20000
Class mapping: {'fake': 1, 'real': 0}


## 5. Model Architecture, Optimizer & Cosine Annealing Scheduler

In [7]:
import timm
import torch.nn as nn

# Load pretrained EfficientNet-B0 with binary head
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
epochs = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

print("Model architecture configured:")
print(f"- Conv Head: {model.conv_head}")
print(f"- Classifier Head: {model.classifier}")

Model architecture configured:
- Conv Head: Conv2d(320, 1280, kernel_size=(1, 1), stride=(1, 1), bias=False)
- Classifier Head: Linear(in_features=1280, out_features=2, bias=True)


## 6. Training & Validation Execution Loop

In [ ]:
import copy
import time
import torch

best_val_acc = 0.0
best_weights = None
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"Starting training for {epochs} epochs on {device}...")
start_time = time.time()

for epoch in range(1, epochs + 1):
    # --- Training Phase ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data).item()
        train_total += inputs.size(0)

    epoch_train_loss = train_loss / max(1, train_total)
    epoch_train_acc = (train_correct / max(1, train_total)) * 100.0

    # --- Validation Phase ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data).item()
            val_total += inputs.size(0)

    epoch_val_loss = val_loss / max(1, val_total)
    epoch_val_acc = (val_correct / max(1, val_total)) * 100.0
    scheduler.step()

    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)

    print(f"Epoch [{epoch:02d}/{epochs:02d}] | Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.2f}%")

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, "deepguard_effnet.pth")
        print(f"  --> New best model checkpoint saved (Val Acc: {best_val_acc:.2f}%)")

total_mins = (time.time() - start_time) / 60.0
print(f"\nTraining complete in {total_mins:.1f} mins. Peak Validation Accuracy: {best_val_acc:.2f}%")

Starting training for 10 epochs on cuda...


## 7. Render Loss and Accuracy Convergence Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_range, history["train_loss"], 'o-', label="Train Loss", color="#6366f1", linewidth=2)
ax1.plot(epochs_range, history["val_loss"], 's--', label="Val Loss", color="#ef4444", linewidth=2)
ax1.set_title("Cross-Entropy Loss Curve", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.plot(epochs_range, history["train_acc"], 'o-', label="Train Accuracy", color="#10b981", linewidth=2)
ax2.plot(epochs_range, history["val_acc"], 's--', label="Val Accuracy", color="#38bdf8", linewidth=2)
ax2.set_title("Classification Accuracy Curve (%)", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=200)
plt.show()

## 8. Export Model for DeepGuard Deployment

Download `deepguard_effnet.pth` and place it into your local repository at:
`backend/weights/deepguard_effnet.pth`

In [ ]:
# If running in Google Colab, trigger automatic browser download:
try:
    from google.colab import files  # type: ignore[import-not-found]  # pyrefly: ignore[missing-import]
    files.download("deepguard_effnet.pth")
    files.download("training_curves.png")
    print("Triggered Colab file download!")
except ImportError:
    print("Local environment detected. Weights are saved at: ./deepguard_effnet.pth")